# MICROPY_* macro survey

Goal: enumerate all `MICROPY_*` macros across the MicroPython codebase, capture definitions/usages, and generate a markdown summary table with module/context hints.

In [25]:
from pathlib import Path
import re
import sqlite3
from textwrap import dedent

ROOT = Path('D:\\mypython\\micropython').resolve()
DB_PATH = ROOT / 'scratch' / 'micropy_macros.db'
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

ROOT


WindowsPath('D:/mypython/micropython')

In [26]:
ignore_parts = {
    'emsdk', 'esp-idf', '.git', '.venv', 'build', 'htmlcov', 'docs',
    'pico-sdk', 'multipage_pair_report', 'obj_compare_out', 'binary_compare_out',
    'myenv', '__pycache__', 'scratch',
}
allowed_names = {'Makefile', 'CMakeLists.txt'}
allowed_suffixes = {
    '.c', '.h', '.cpp', '.hpp', '.mk', '.cmake', '.ld', '.py', '.rst',
    '.inc', '.S', '.s', '.txt'
}

def is_code_file(path: Path) -> bool:
    if path.is_dir():
        return False
    if any(part in ignore_parts for part in path.parts):
        return False
    if path.name in allowed_names:
        return True
    return path.suffix in allowed_suffixes

code_files = [p for p in ROOT.rglob('*') if is_code_file(p)]
len(code_files), code_files[:3]

(15868,
 [WindowsPath('D:/mypython/micropython/examples/accellog.py'),
  WindowsPath('D:/mypython/micropython/examples/accel_i2c.py'),
  WindowsPath('D:/mypython/micropython/examples/asmled.py')])

In [27]:
cur.executescript("""
DROP TABLE IF EXISTS macros;
DROP TABLE IF EXISTS occurrences;

CREATE TABLE macros (
    name TEXT PRIMARY KEY,
    file TEXT,
    line INTEGER,
    value TEXT,
    comment TEXT,
    description TEXT,
    kind TEXT,
    description_ai TEXT
);

CREATE TABLE occurrences (
    name TEXT,
    file TEXT,
    line INTEGER,
    snippet TEXT
);
""")
conn.commit()
DB_PATH

WindowsPath('D:/mypython/micropython/scratch/micropy_macros.db')

In [28]:
define_re = re.compile(r"^\s*#\s*define\s+(MICROPY_[A-Z0-9_]+)(?:\s+(.*))?$")


def extract_leading_comment(lines, idx):
    comment_lines = []
    j = idx - 2  # zero-based index for list
    while j >= 0:
        stripped = lines[j].strip()
        if stripped.startswith('//') or stripped.startswith('/*') or stripped.startswith('*'):
            comment_lines.insert(0, stripped.strip('/ *'))
            j -= 1
            continue
        break
    return ' '.join(comment_lines)


def better_def(existing, candidate):
    # Prefer longer comment; if equal, prefer longer value; else keep existing
    if existing is None:
        return candidate
    if len(candidate['comment']) > len(existing['comment']):
        return candidate
    if len(candidate['comment']) == len(existing['comment']) and len(candidate['value']) > len(existing['value']):
        return candidate
    return existing


def collect_definitions():
    best_by_name = {}
    for path in code_files:
        try:
            text = path.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        lines = text.splitlines()
        for idx, line in enumerate(lines, start=1):
            m = define_re.match(line)
            if not m:
                continue
            name = m.group(1)
            value = (m.group(2) or '').strip()
            comment = extract_leading_comment(lines, idx)
            kind = 'define'
            rec = {
                'name': name,
                'value': value,
                'comment': comment,
                'description': comment,
                'file': path.relative_to(ROOT).as_posix(),
                'line': idx,
                'kind': kind,
                'description_ai': None,
            }
            best_by_name[name] = better_def(best_by_name.get(name), rec)

    rows = [
        (
            r['name'],
            r['file'],
            r['line'],
            r['value'],
            r['comment'],
            r['description'],
            r['kind'],
            r['description_ai'],
        )
        for r in best_by_name.values()
    ]
    cur.executemany(
        """
        INSERT OR REPLACE INTO macros
            (name, file, line, value, comment, description, kind, description_ai)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        rows,
    )
    conn.commit()
    return len(rows)

collect_definitions()

1793

In [29]:
macro_use_re = re.compile(r"MICROPY_[A-Z0-9_]+")

def collect_occurrences():
    rows = []
    for path in code_files:
        try:
            text = path.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        lines = text.splitlines()
        for idx, line in enumerate(lines, start=1):
            matches = macro_use_re.findall(line)
            if not matches:
                continue
            snippet = line.strip()[:240]
            rows.extend((name, str(path.relative_to(ROOT)), idx, snippet) for name in matches)
    cur.executemany(
        "INSERT INTO occurrences (name, file, line, snippet) VALUES (?, ?, ?, ?)",
        rows,
    )
    conn.commit()
    return len(rows)

collect_occurrences()

26460

In [ ]:
from collections import defaultdict, Counter
from macro_utils import module_hint, macro_category, choose_best_description



def build_summary(limit_modules: int = 5):
    defs = cur.execute(
        "SELECT name, value, comment, description, file, line FROM macros"
    ).fetchall()
    occ = cur.execute(
        "SELECT name, file, line FROM occurrences"
    ).fetchall()

    mod_counts = defaultdict(Counter)
    files_per_macro = defaultdict(set)
    for name, file, line in occ:
        mod_counts[name][module_hint(file)] += 1
        files_per_macro[name].add(file)

    defs_by_name = defaultdict(list)
    for name, value, comment, description, file, line in defs:
        defs_by_name[name].append({
            'value': value,
            'comment': comment,
            'description': description,
            'file': file,
            'line': line,
        })

    summary = []
    for name in sorted(defs_by_name.keys()):
        best_desc, best_loc = choose_best_description(defs_by_name[name])
        modules = ', '.join(
            f"{mod} ({count})" for mod, count in mod_counts[name].most_common(limit_modules)
        )
        summary.append(
            {
                'name': name,
                'category': macro_category(name),
                'description': best_desc.strip(),
                'value': defs_by_name[name][0].get('value', ''),
                'modules': modules,
                'definition': best_loc,
                'files_touched': len(files_per_macro.get(name, [])),
            }
        )
    return summary

summary = build_summary()
len(summary)

ModuleNotFoundError: No module named 'macro_utils'

In [31]:
def render_markdown(summary, min_group_size: int = 11):
    header = "| Macro | Description | Value | Top Modules (uses) | Defined at |\n"
    header += "|---|---|---|---|---|\n"

    grouped = defaultdict(list)
    for item in summary:
        grouped[item['category']].append(item)

    misc_bucket = []
    filtered = {}
    for category, items in grouped.items():
        if len(items) < min_group_size:
            misc_bucket.extend(items)
        else:
            filtered[category] = items
    if misc_bucket:
        filtered['MISC'] = filtered.get('MISC', []) + misc_bucket

    sections = []
    for category in sorted(filtered.keys()):
        rows = []
        for item in sorted(filtered[category], key=lambda x: x['name']):
            desc = item['description']
            val = item['value'] or '—'
            modules = item['modules'] or '—'
            rows.append(
                f"| `{item['name']}` | {desc.replace('|', '\\|')} | {val.replace('|', '\\|')} | {modules} | {item['definition']} |"
            )
        section_md = header + "\n".join(rows)
        sections.append(f"\n\n### {category}\n\n" + section_md)

    md = "\n".join(sections)
    md_path = ROOT / 'scratch' / 'MICROPY_macros.md'
    md_path.write_text(md)
    return md_path

md_path = render_markdown(summary)


## Outputs
- SQLite database: `scratch/micropy_macros.db` (definitions + occurrences)
- Markdown summary: `scratch/MICROPY_macros.md` (table of macros, comments/values, modules, definition site)

Run the cells above to refresh the data after repo updates.

## AI enrichment pipeline (Azure OpenAI)

Configure the Azure OpenAI endpoint/key/model via env vars before running the cells below:

- `AZURE_OPENAI_ENDPOINT`

- `AZURE_OPENAI_KEY` (if key auth is enabled)

- `AZURE_OPENAI_MODEL` (e.g., `gpt-4o-mini`)

- `AZURE_OPENAI_TOKEN` (bearer token) **or** Azure AD credentials for `DefaultAzureCredential`



Required Python  Modules:

- `httpx`

- `azure-identity` (only if using Azure AD / managed identity)



The code will:

1. Ensure a `description_ai` column exists in `macros`.

2. Select macros with empty/short descriptions.

3. Build context-rich prompts per macro.

4. Call Azure OpenAI in batches to suggest one-line descriptions.

5. Store results in SQLite and regenerate the markdown using AI text when present.


In [32]:
import os
from typing import List, Dict, Any

# Fetch candidates with empty/short descriptions
MIN_LEN = 15
candidates = cur.execute(
    """
    SELECT m.name, m.value, COALESCE(m.description, m.comment, '') as description, m.comment, m.file, m.line
    FROM macros m
    WHERE (m.description_ai IS NULL OR length(m.description_ai)=0)
      AND (m.description IS NULL OR length(m.description) < ?)
    LIMIT 2000
    """,
    (MIN_LEN,),
).fetchall()
len(candidates)

1539

In [33]:
import dotenv
dotenv.load_dotenv()

AZ_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', 'https://mpy-foundry.cognitiveservices.azure.com')
AZ_KEY = os.getenv('AZURE_OPENAI_KEY')  # optional if using key auth
AZ_MODEL = os.getenv('AZURE_OPENAI_MODEL', 'gpt-4o-mini')
AZ_TOKEN = os.getenv('AZURE_OPENAI_TOKEN')  # bearer token from AAD or user-provided
AZ_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-05-01-preview')


In [34]:
SYSTEM_PROMPT = """
You are documenting MicroPython build-time macros. Stay within provided context.
If the purpose is unclear, respond with "unknown". Output a concise discription of one or two lines.
""".strip()

import httpx
import json
from collections import Counter
from hashlib import sha256

from azure.identity import DefaultAzureCredential  # optional; only needed for AAD/managed identity
from diskcache import Cache

TIMEOUT = 60
API_VERSION = AZ_API_VERSION
OCCURRENCE_LIMIT = 8
OCCURRENCE_RADIUS = 3
AI_TEMPERATURE = 0.01

CACHE_PATH = ROOT / 'scratch' / 'ai_cache'
cache = Cache(str(CACHE_PATH))


def top_modules(name: str, limit: int = 3) -> str:
    rows = cur.execute("SELECT file FROM occurrences WHERE name=?", (name,)).fetchall()
    counter = Counter(module_hint(r[0]) for r in rows)
    return ', '.join(f"{m} ({c})" for m, c in counter.most_common(limit))


def get_occurrence_contexts(name: str, radius: int = OCCURRENCE_RADIUS, limit: int = OCCURRENCE_LIMIT) -> str:
    rows = cur.execute(
        "SELECT file, line FROM occurrences WHERE name=? LIMIT ?",
        (name, limit),
    ).fetchall()
    blocks = []
    for file, line in rows:
        path = ROOT / file
        try:
            text = path.read_text(encoding='utf-8', errors='ignore')
            lines = text.splitlines()
            start = max(0, line - 1 - radius)
            end = min(len(lines), line + radius)
            snippet_lines = []
            for idx in range(start, end):
                snippet_lines.append(f"{idx + 1}: {lines[idx]}")
            blocks.append(f"{file}:{line}\n" + "\n".join(snippet_lines))
        except Exception:
            continue
    return "\n\n".join(blocks)


def build_user_prompt(rec: Dict[str, Any]) -> str:
    name = rec['name']
    value = rec.get('value') or ''
    description = rec.get('description') or ''
    comment = rec.get('comment') or ''
    loc = f"{rec['file']}:{rec['line']}"
    modules = rec.get('modules', '')
    occurrences = rec.get('occurrences', '')
    return dedent(f"""
    Macro: {name}
    Value: {value}
    Existing description/comment: {description or comment}
    Location: {loc}
    Top modules: {modules}
    Occurrences (±{OCCURRENCE_RADIUS} lines):
    {occurrences}
    Task: One-line purpose; if unclear, reply "unknown".
    Return JSON: {{"name": "{name}", "description": "..."}}
    """)


def make_cache_key(rec: Dict[str, Any]) -> str:
    parts = [
        AZ_MODEL or '',
        rec.get('name', ''),
        rec.get('value', ''),
        rec.get('description', ''),
        rec.get('comment', ''),
        rec.get('occurrences', ''),
    ]
    raw = "\u0001".join(parts)
    return sha256(raw.encode('utf-8')).hexdigest()


# Build enriched candidate records
cand_rows: List[Dict[str, Any]] = []
for name, value, description, comment, file, line in candidates:
    cand_rows.append(
        {
            'name': name,
            'value': value,
            'description': description,
            'comment': comment,
            'file': file,
            'line': line,
            'modules': top_modules(name, limit=3),
            'occurrences': get_occurrence_contexts(name),
        }
    )

print(f"Candidates needing AI description: {len(cand_rows)}")

_credential = None


def get_auth_headers():
    global _credential
    if AZ_TOKEN:
        return {"Authorization": f"Bearer {AZ_TOKEN}", "Content-Type": "application/json"}
    if AZ_KEY:
        return {"api-key": AZ_KEY, "Content-Type": "application/json"}
    if DefaultAzureCredential is not None:
        if _credential is None:
            _credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
        token = _credential.get_token("https://cognitiveservices.azure.com/.default")
        return {"Authorization": f"Bearer {token.token}", "Content-Type": "application/json"}
    print("No authentication configured (set AZURE_OPENAI_TOKEN, AZURE_OPENAI_KEY, or enable DefaultAzureCredential).")
    return None


def call_azure_one(rec: Dict[str, Any]) -> str:
    cache_key = make_cache_key(rec)
    cached = cache.get(cache_key)
    if cached is not None:
        return str(cached)

    if not AZ_ENDPOINT:
        return ''
    headers = get_auth_headers()
    if not headers:
        return ''
    url = f"{AZ_ENDPOINT}/openai/deployments/{AZ_MODEL}/chat/completions?api-version={API_VERSION}"
    with httpx.Client(timeout=TIMEOUT) as client:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(rec)},
        ]
        resp = client.post(
            url,
            headers=headers,
            json={
                "messages": messages,
                "temperature": AI_TEMPERATURE,
                "response_format": {"type": "json_object"},
            },
        )
        resp.raise_for_status()
        content = resp.json()["choices"][0]["message"]["content"]
        try:
            data = json.loads(content)
            desc = (data.get("description") or '').strip()
        except Exception:
            desc = ''

    cache.set(cache_key, desc)
    return desc


updated = 0
progress_bar = ''
if AZ_ENDPOINT and cand_rows:
    total = len(cand_rows)
    for idx, rec in enumerate(cand_rows, start=1):
        try:
            desc = call_azure_one(rec)
        except Exception as e:
            print(f"\nError processing {rec['name']}: {e}")
            continue
        if desc and desc.lower() != 'unknown':
            cur.execute(
                "UPDATE macros SET description_ai=?, description=? WHERE name=?",
                (desc, desc, rec['name']),
            )
            conn.commit()
            updated += 1
        progress_bar += '>'
        print(f"{progress_bar} {idx}/{total}", end='\r', flush=True)
        if len(progress_bar) > 50:
            progress_bar = ""
            print("\n")
    print(f"\nAI descriptions updated: {updated}/{total}")
else:
    print("Skipping Azure calls (missing endpoint or no candidates).")


ModuleNotFoundError: No module named 'diskcache'

In [ ]:
# Regenerate summary/markdown preferring description_ai when available

def choose_best_description(defs_for_name):
    ai = [d for d in defs_for_name if d.get('description_ai')]
    if ai:
        ai.sort(key=lambda d: len(d['description_ai']), reverse=True)
        return ai[0]['description_ai']
    descr = [d for d in defs_for_name if d.get('description')]
    if descr:
        descr.sort(key=lambda d: len(d['description']), reverse=True)
        return descr[0]['description']
    comment_candidates = [d for d in defs_for_name if d.get('comment')]
    if comment_candidates:
        comment_candidates.sort(key=lambda d: len(d['comment']), reverse=True)
        return comment_candidates[0]['comment']
    value_candidates = [d for d in defs_for_name if d.get('value')]
    if value_candidates:
        value_candidates.sort(key=lambda d: len(d['value']), reverse=True)
        return value_candidates[0]['value']
    return ''


def build_summary_with_ai(limit_modules: int = 5):
    defs = cur.execute(
        "SELECT name, value, comment, description, description_ai, file, line FROM macros"
    ).fetchall()
    occ = cur.execute(
        "SELECT name, file, line FROM occurrences"
    ).fetchall()

    mod_counts = defaultdict(Counter)
    files_per_macro = defaultdict(set)
    for name, file, line in occ:
        mod_counts[name][module_hint(file)] += 1
        files_per_macro[name].add(file)

    defs_by_name = defaultdict(list)
    for name, value, comment, description, description_ai, file, line in defs:
        defs_by_name[name].append({
            'value': value,
            'comment': comment,
            'description': description,
            'description_ai': description_ai,
            'file': file,
            'line': line,
        })

    summary = []
    for name in sorted(defs_by_name.keys()):
        best_desc = (choose_best_description(defs_by_name[name]) or '').strip()
        modules = ', '.join(
            f"{mod} ({count})" for mod, count in mod_counts[name].most_common(limit_modules)
        )
        best_loc = f"{defs_by_name[name][0]['file']}:{defs_by_name[name][0]['line']}"
        summary.append(
            {
                'name': name,
                'category': macro_category(name),
                'description': best_desc,
                'value': defs_by_name[name][0].get('value', ''),
                'modules': modules,
                'definition': best_loc,
                'files_touched': len(files_per_macro.get(name, [])),
            }
        )
    return summary


def render_markdown_with_ai(summary, min_group_size: int = 11):
    header = "| Macro | Description | Value | Top Modules (uses) | Defined at |\n"
    header += "|---|---|---|---|---|\n"

    grouped = defaultdict(list)
    for item in summary:
        grouped[item['category']].append(item)

    misc_bucket = []
    filtered = {}
    for category, items in grouped.items():
        if len(items) < min_group_size:
            misc_bucket.extend(items)
        else:
            filtered[category] = items
    if misc_bucket:
        filtered['MISC'] = filtered.get('MISC', []) + misc_bucket

    sections = []
    for category in sorted(filtered.keys()):
        rows = []
        for item in sorted(filtered[category], key=lambda x: x['name']):
            desc = item['description']
            val = item['value'] or '—'
            modules = item['modules'] or '—'
            rows.append(
                f"| `{item['name']}` | {desc.replace('|', '\\|')} | {val.replace('|', '\\|')} | {modules} | {item['definition']} |"
            )
        section_md = header + "\n".join(rows)
        sections.append(f"\n\n### {category}\n\n" + section_md)

    md = "\n".join(sections)
    md_path = ROOT / 'scratch' / 'MICROPY_macros.md'
    md_path.write_text(md)
    return md_path

summary_ai = build_summary_with_ai()
md_path = render_markdown_with_ai(summary_ai)
md_path